In [0]:
from pyspark.sql.functions import sum,count,when,lit,col,countDistinct,avg,max,row_number
from pyspark.sql.window import Window
from pyspark import pipelines as dp

In [0]:
@dp.materialized_view(
    name = "restaurant.gold.d_customers_360_py",
    table_properties = {"quality":"gold"}
)
def customers_360():
    df_orders = (
             spark.table("restaurant.silver.fact_orders")
                  .groupBy(col("customer_id"))
                  .agg(
                       countDistinct(col("order_id")).alias("total_orders"),
                       sum(col("total_amount")).cast("decimal(10,2)").alias("lifetime_spend"),
                       avg(col("total_amount")).cast("decimal(10,2)").alias("avg_order_value"),
                       max(col("order_date")).alias("last_order_date"),
                       when(col("lifetime_spend") >= 5000,"Platinum").when(col("lifetime_spend") >= 3000,"Gold").when(col("lifetime_spend") >= 1000,"Silver").otherwise("bronze").alias("loyality_tier")
                  ))
    df_reviews = (
              spark.table("restaurant.silver.fact_reviews")
                   .groupBy("customer_id")
                   .agg(
                       avg(col("rating")).cast("decimal (3,2)").alias("avg_rating_given"),
                       countDistinct(col("review_id")).alias("total_reviews")
                   ))
    df_fact_orders = spark.table("restaurant.silver.fact_orders")
    df_fact_items = spark.table("restaurant.silver.fact_order_items")

    df_fav_item = (
                   df_fact_orders.alias("o").join(df_fact_items.alias("i"),"order_id")
                              .groupBy(col("o.customer_id"),col("i.item_name"))
                              .agg(sum(col("i.quantity")).alias("total_qty"))
                              .withColumn("rnk",row_number().over(Window.partitionBy(col("o.customer_id")).orderBy(col("total_qty").desc())))
                              .filter(col("rnk") == 1)
                              .select(col("o.customer_id"),col("i.item_name"))
                    )
    window = Window.partitionBy(col("customer_id")).orderBy(col("total_orders").desc())

    df_rest = spark.table("restaurant.silver.dim_restaurants")
    df_fav_rest = (
               df_rest.join(df_fact_orders,"restaurant_id")
                      .groupBy("customer_id","name")
                      .agg(countDistinct(col("order_id")).alias("total_orders"))
                      .withColumn("rnk",row_number().over(window))
                      .filter(col("rnk") == 1)
                      .select("customer_id",col("name").alias("restaurant_name"))
               )
    df_customers = (
                spark.table("restaurant.silver.dim_customers").alias("c")
                     .join(df_orders,"customer_id","left")
                     .join(df_reviews,"customer_id","left")
                     .join(df_fav_item,"customer_id","left")
                     .join(df_fav_rest,"customer_id","left")
                     .select(col("c.customer_id"),col("c.name").alias("customer_name"),col("c.email"),col("c.city"),col("c.join_date"),col("total_orders"),col("lifetime_spend"),col("avg_order_value"),col("loyality_tier"),col("last_order_date"),col("avg_rating_given"),col("total_reviews"),col("restaurant_name").alias("favorite_restaurant"),col("item_name").alias("favorite_item"))
                    )
    return df_customers
